In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor


In [3]:

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")


In [4]:
from openai import OpenAI
from dotenv import load_dotenv
from gitsource import GithubRepositoryDataReader
from minsearch import Index

from rag_helper import RAGBase

COMMIT = "8c1834d"
load_dotenv()

    

True

In [5]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id=COMMIT,
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [6]:
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

client = OpenAI()
rag = RAGBase(index=index, llm_client=client)

In [7]:
class RAGTraced(RAGBase):

    def search(self, query):
        with tracer.start_as_current_span("search"):
            return super().search(query)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as llm:
            response = super().llm(prompt)
            llm.set_attribute("input_tokens",response.usage.input_tokens)
            llm.set_attribute("output_tokens",response.usage.output_tokens)
            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

In [8]:
rag_traced = RAGTraced(
    rag.index,
    rag.llm_client,
    rag.model
)

In [9]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = rag_traced.rag(query)

print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x0da771d29bb5ea139a1a420e0e377129",
        "span_id": "0x52b2452db70252a4",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x2a77f68b3e7faa0f",
    "start_time": "2026-07-17T18:35:58.453760Z",
    "end_time": "2026-07-17T18:35:58.459678Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {},
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "5369f54e-e4b1-46c9-95ce-ba6543acc24b",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}


{
    "name": "llm",
    "context": {
        "trace_id": "0x0da771d29bb5ea139a1a420e0e377129",
        "span_id": "0xa60fb410fe156ffe",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x2a77f68b3e7faa0f",
    "start_time": "2026-07-17T18:35:58.460871Z",
    "end_time": "2026-07-17T18:36:03.332276Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "input_tokens": 7070,
        "output_tokens": 290
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "5369f54e-e4b1-46c9-95ce-ba6543acc24b",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "rag",
    "context": {
        "trace_id": "0x0da771d29bb5ea139a1a420e0e377129",
        "span_id": "0x2a77f68b3e7faa